# 12 — Recurrent Successor-Feature agent with an implicit context state (`recurrent_sf`)

Implements [`docs/recurrent_sf_context_plan.md`](../docs/recurrent_sf_context_plan.md).

**The idea in one paragraph.** In the corner maze, reward is *regime-gated*: during
`PRETRIAL` the agent is walled into the start arm and no goal reward exists; only after
it holds the pretrial trigger for `PRETRIAL_MIN_STEPS` (=7) do the barriers open, the
phase flips to `TRIAL`, and the goal well starts to pay out. **This phase is hidden from
the observation** — the same pose recurs across pretrial / trial / iti with different
reward implications. A memoryless or movement-only agent therefore cannot tell whether a
location is rewarding. This model gives the agent a **GRU context state `h_t`** that folds
in *observation, previous action, and previous reward* to infer the hidden reward-machine
state, and a **single action-conditioned successor-feature head `ψ`** re-weighted by a
**context-dependent reward vector `w(z_t)`** — no bank of switched maps. It supersedes the
online-tabular SR in `models/sr.py` that [fails on this task](../md/sr-yoked-negative-results.md).

**What this notebook does** (milestone ladder from the plan §11):
- **M0** build + smoke: the model instantiates, forward pass gives the §4.4 shapes, one
  TBPTT step runs, and the episode-boundary GRU reset is unit-tested.
- **M1/M2** train the full model on one acquisition `session_type`; show that a linear probe
  reads the hidden phase out of `h_t` with **low error**, while a memoryless baseline sits
  near chance — the empirical statement that *history is required and `h_t` carries the context*.
- **M3** the six-arm baseline matrix (full vs memoryless / no-reward / oracle / stack / switched-maps).

> **Note on scope & honesty.** Learning the full 32-trial acquisition to criterion from scratch
> is a hard sparse-reward RL problem; a from-scratch SF agent will not necessarily *solve*
> navigation within a laptop-overnight budget. The **representational** result (does `h_t`
> encode the context) is robust at modest budgets and is the scientific headline here. The
> behavioral metrics are reported honestly, whatever the policy achieves. Scale knobs are in §1.

## 0. Colab bootstrap

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ryangrg/corner-maze-rl/blob/main/notebooks/12_RSF.ipynb)

Run the cell below first. Locally it is a no-op. On Colab it clones the repo, `pip install -e .`
(brings the lookups: `grid_cells_60d.npz`, `pose_visual.npz`), and starts an optional
background Drive sync of `data/runs/`. CPU is fine for this notebook — the model is small.

In [ ]:
# ── Colab bootstrap (no-op locally; local-write + periodic Drive sync) ─
import sys
IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    import os, subprocess, threading, atexit
    from pathlib import Path

    REPO_URL          = 'https://github.com/ryangrg/corner-maze-rl.git'
    REPO_BRANCH       = 'main'
    REPO_DIR          = Path('/content/corner-maze-rl')
    USE_DRIVE         = True
    DRIVE_ROOT        = Path('/content/drive/MyDrive/corner-maze-rl-colab')
    SYNC_INTERVAL_SEC = 120

    if not REPO_DIR.exists():
        subprocess.check_call(['git', 'clone', '--depth=1',
                               '-b', REPO_BRANCH, REPO_URL, str(REPO_DIR)])
    else:
        print(f'  repo already at {REPO_DIR}')

    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)])

    src_dir = str(REPO_DIR / 'src')
    if src_dir not in sys.path:
        sys.path.insert(0, src_dir)

    local_runs = REPO_DIR / 'data' / 'runs'
    local_runs.mkdir(parents=True, exist_ok=True)

    drive_runs   = None
    drive_status = 'skipped (USE_DRIVE=False)'
    if USE_DRIVE:
        try:
            from google.colab import drive
            drive.mount('/content/drive')
            if not DRIVE_ROOT.exists():
                drive_status = f'mounted, but {DRIVE_ROOT} missing — sync disabled'
            else:
                drive_runs = DRIVE_ROOT / 'runs'
                drive_runs.mkdir(exist_ok=True)
                drive_status = f'mounted at {DRIVE_ROOT}'
        except Exception as e:
            drive_status = f'mount failed ({type(e).__name__}) — sync disabled'

    _sync_lock = threading.Lock()
    def sync_runs_to_drive(verbose: bool = False):
        if drive_runs is None:
            return (False, 'no drive target')
        if not _sync_lock.acquire(blocking=False):
            return (False, 'sync already in flight')
        try:
            result = subprocess.run(
                ['rsync', '-a', '--update', str(local_runs) + '/', str(drive_runs) + '/'],
                capture_output=True, text=True, timeout=300)
            ok  = (result.returncode == 0)
            msg = 'synced' if ok else f'rc={result.returncode}: {result.stderr.strip()[:200]}'
            if verbose: print(f'  [drive sync] {msg}')
            return (ok, msg)
        except Exception as e:
            if verbose: print(f'  [drive sync] {type(e).__name__}: {e}')
            return (False, f'{type(e).__name__}: {e}')
        finally:
            _sync_lock.release()

    _sync_stop = threading.Event()
    def _sync_loop():
        while not _sync_stop.wait(SYNC_INTERVAL_SEC):
            sync_runs_to_drive(verbose=False)
    if drive_runs is not None:
        threading.Thread(target=_sync_loop, daemon=True).start()
        atexit.register(lambda: (_sync_stop.set(), sync_runs_to_drive(verbose=True)))

    os.chdir(REPO_DIR / 'notebooks')
    print(f'\n✓ Colab bootstrap complete — cwd={Path.cwd()}  drive={drive_status}')
else:
    print('✓ local environment — bootstrap skipped')

## 1. Configuration

Every non-obvious choice from the plan §3 / §6.4 is a knob on `RecurrentSFConfig`, defaulted
to the plan's decision with the rationale inline. `SCALE` picks a preset:

| `SCALE` | what it's for | rough CPU time |
|---|---|---|
| `'smoke'` | prove every code path runs | ~1 min |
| `'demo'`  | **default** — clear phase-probe separation across arms | ~5–10 min |
| `'headline'` | more seeds / episodes for CI'd figures | ~1 hr+ |

`SESSION_TYPE_ACQ` is the acquisition context. Default **`'PI+VC f2 acquisition'`** has a
**visible cue** that reveals the goal, so the agent can decode the goal from the observation and
the GRU only has to infer the *phase* (still hidden) — an easier starting point. `'PI acquisition'`
(no cue) forces the GRU to carry the goal too — the memory-stress variant, explored later (§13).

In [ ]:
from __future__ import annotations
import copy, json, math, time
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from corner_maze_rl.encoders.grid_cells import GridCellEncoder
from corner_maze_rl.encoders.state_vectors import (
    LOCATION_TO_INDEX, NUM_LOCATIONS,
    PHASE_ITI, PHASE_PRETRIAL, PHASE_TRIAL, _STATE_TYPE_TO_PHASE)
from corner_maze_rl.env.constants import (
    WELL_REWARD_SCR, WELL_LOCATIONS, PRETRIAL_MIN_STEPS,
    CRITERION_SCORE_THRESHOLD, CRITERION_CONSECUTIVE_REQUIRED)
from corner_maze_rl.env.corner_maze_env import CornerMazeEnv
from corner_maze_rl.models.base import TrainableAgent
from corner_maze_rl.train.runner import run_session_sequence

# ── shape constants (plan §4.4) ─────────────────────────────────
PHI_DIM      = NUM_LOCATIONS * 4     # 196 one-hot pose basis (49 floor cells × 4 dirs)
NUM_ACTIONS  = 5                     # L, R, F, enter-well, pause
POSE_ENC_DIM = 60                    # grid-cell pose encoder e(pose)
OBS_DIM      = 60                    # embedding obs o_t
NUM_PHASES   = 3                     # {trial, pretrial, iti}
assert PHI_DIM == 196

# ── experiment identity ─────────────────────────────────────────
RUN_NAME          = 'recurrent-sf-pivc-f2'
EXPERIMENT_NOTES  = ('Recurrent SF agent w/ implicit context; acquisition = PI+VC f2 '
                     '(visible cue → goal is decodable from the observation, so the GRU '
                     'leans less on memory to infer context; phase is still hidden). '
                     'PI (no cue) is the memory-stress variant — see §13 recipes.')

# ── env paradigm (plan §8) ──────────────────────────────────────
# Valid acquisition session_types: 'PI acquisition', 'PI+VC f2 acquisition',
#   'PI+VC f1 acquisition', 'VC acquisition'.  Probe families (optional, only
#   run if acquisition reaches criterion): 'PI novel route no cue',
#   'PI reversal no cue', ...  (see env/corner_maze_env.py SESSION_GENERATORS)
SESSION_TYPE_ACQ    = 'PI+VC f2 acquisition'   # visible-cue context (cue reveals the goal);
                                               # 'PI acquisition' (no cue) needs stronger memory.
PROBE_SESSION_TYPES = []             # e.g. ['PI+VC f2 novel route', 'PI+VC f2 reversal']
ORIENTATION         = 'N/NE'
GOAL_LOCATION       = 'NE'

# ── which baseline arms to run in §10 (plan §9) ─────────────────
#  'full'      — the model
#  'B1_memoryless' — GRU_H=0 (z = e(pose) only)
#  'B5_no_reward'  — GRU input = [o, a_{t-1}] only (no reward history)
#  'B3_oracle'     — true phase one-hot concatenated to z (upper bound)
#  'B2_stack'      — finite history stack of last k (o,a,r)
#  'B4_switched'   — bank of K {ψ,w} heads + soft gate (the "to-beat" baseline)
ARMS = ['full', 'B1_memoryless', 'B5_no_reward', 'B3_oracle']   # + 'B2_stack','B4_switched'

# ── device ──────────────────────────────────────────────────────
FORCE_CPU = True   # model is tiny; CPU is fastest & most portable here
if FORCE_CPU:
    DEVICE = 'cpu'
elif torch.cuda.is_available():
    DEVICE = 'cuda'
else:
    DEVICE = 'cpu'


@dataclass
class RecurrentSFConfig:
    """Every knob from plan §3 / §6.4. Change a knob → re-run."""
    # architecture
    memory: str = 'gru'              # 'gru' | 'stack_k' | 'none' (none ⇒ B1 memoryless)
    gru_h: int = 128                 # H (knob 4); 0 also ⇒ memoryless
    stack_k: int = 8                 # history length for memory='stack_k' (B2)
    gru_input: str = 'obs_act_rew'   # 'obs_act_rew' | 'no_reward' (B5)
    ctx_input: str = 'none'          # 'none' | 'oracle'  (B3 concats true phase to z)
    switched_maps: bool = False      # B4: bank of K {ψ,w} heads + soft gate
    n_maps: int = 3
    tie_reward_head: bool = True     # r̂ = φ·w (knob 7)
    hidden: int = 256                # head MLP width
    # SF / RL
    gamma: float = 0.99              # knob 5
    lr: float = 3e-4
    grad_clip: float = 1.0
    tau: float = 0.005               # Polyak target update
    # losses (plan §5)
    lambda_psi: float = 1.0
    lambda_r: float = 1.0
    lambda_trans: float = 0.5
    lambda_ctx: float = 0.5          # annealed → lambda_ctx_floor
    lambda_ctx_floor: float = 0.05
    ctx_anneal_updates: int = 2000
    # w-regression phase filter (plan §5 loss 2 / knob 11). Plan default = {TRIAL}.
    # We read `phase` BEFORE stepping, so the goal reward (returned by that step) is
    # attributed to the TRIAL-phase record that earned it — {TRIAL} DOES capture every
    # goal reward (no "reward gets dropped" trap). Alternative: add PRETRIAL/ITI so w
    # also learns r≈0 there (makes the reward gating explicit). Exposure (all-wells) is
    # never trained, so neither choice risks static-w contamination. See §5 markdown.
    w_from_phases: tuple = (PHASE_TRIAL,)
    # TBPTT / replay (knob 6)
    tbptt_l: int = 32
    batch_windows: int = 8
    buffer_capacity: int = 200
    min_episodes_to_update: int = 8      # plan §6.4 (presets shrink this for faster smoke/demo)
    updates_per_collect: int = 4
    # exploration (knob 9)
    temp_start: float = 1.0
    temp_end: float = 0.1
    temp_anneal_updates: int = 2000
    eps_floor: float = 0.05
    # reward scaling (knob 10): 'fixed' | 'none' | 'return_std' (see §5 note)
    reward_norm: str = 'fixed'
    # misc
    device: str = DEVICE
    seed: int = 0

    @property
    def use_memory(self) -> bool:
        return self.memory != 'none' and self.gru_h > 0
    @property
    def h_dim(self) -> int:
        return self.gru_h if self.use_memory else 0
    @property
    def z_dim(self) -> int:
        d = POSE_ENC_DIM + self.h_dim
        if self.ctx_input == 'oracle':
            d += NUM_PHASES
        return d
    @property
    def gru_in_dim(self) -> int:
        base = OBS_DIM + NUM_ACTIONS
        if self.gru_input != 'no_reward':
            base += 1
        return base


# ── arm → config delta (plan §9) ────────────────────────────────
def arm_config(arm: str, seed: int, **over) -> RecurrentSFConfig:
    base = dict(seed=seed, **over)
    if arm == 'full':            d = {}
    elif arm == 'B1_memoryless': d = dict(memory='none', gru_h=0)
    elif arm == 'B5_no_reward':  d = dict(gru_input='no_reward')
    elif arm == 'B3_oracle':     d = dict(ctx_input='oracle')
    elif arm == 'B2_stack':      d = dict(memory='stack_k')
    elif arm == 'B4_switched':   d = dict(switched_maps=True)
    else: raise ValueError(f'unknown arm {arm!r}')
    return RecurrentSFConfig(**{**base, **d})


# ── scale presets ───────────────────────────────────────────────
SCALE = 'demo'   # 'smoke' | 'demo' | 'headline'
PRESETS = {
    'smoke':    dict(N_SEEDS=1, MAX_ACQ_EPISODES=8,   BASELINE_EPISODES=12,  MAX_STEPS=200,
                     cfg=dict(tbptt_l=16, batch_windows=4, min_episodes_to_update=2, updates_per_collect=2)),
    'demo':     dict(N_SEEDS=2, MAX_ACQ_EPISODES=120, BASELINE_EPISODES=120, MAX_STEPS=250,
                     cfg=dict(tbptt_l=24, batch_windows=8, min_episodes_to_update=4, updates_per_collect=4)),
    'headline': dict(N_SEEDS=5, MAX_ACQ_EPISODES=400, BASELINE_EPISODES=300, MAX_STEPS=400,
                     cfg=dict(tbptt_l=32, batch_windows=8, min_episodes_to_update=6, updates_per_collect=4)),
}
_P = PRESETS[SCALE]
N_SEEDS           = _P['N_SEEDS']
MAX_ACQ_EPISODES  = _P['MAX_ACQ_EPISODES']   # cap so a non-acquiring run terminates
BASELINE_EPISODES = _P['BASELINE_EPISODES']  # per-arm episodes in the §10 matrix
MAX_STEPS         = _P['MAX_STEPS']          # env max_steps per episode
CFG               = RecurrentSFConfig(**_P['cfg'])   # base config for the full model

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if not (REPO_ROOT / 'data' / 'lookups').exists():   # robust to cwd: locate via installed package
    try:
        import corner_maze_rl as _cmr
        REPO_ROOT = Path(_cmr.__file__).resolve().parents[2]
    except Exception:
        pass
RUN_ROOT  = REPO_ROOT / 'data' / 'runs' / f'{RUN_NAME}_{SCALE}'

print(f'SCALE={SCALE}  N_SEEDS={N_SEEDS}  MAX_ACQ_EPISODES={MAX_ACQ_EPISODES}  MAX_STEPS={MAX_STEPS}')
print(f'session(acq)={SESSION_TYPE_ACQ!r}  probes={PROBE_SESSION_TYPES}  arms={ARMS}')
print(f'device={DEVICE}  z_dim={CFG.z_dim}  gru_in={CFG.gru_in_dim}  H={CFG.h_dim}')
print(f'criterion={CRITERION_SCORE_THRESHOLD} x{CRITERION_CONSECUTIVE_REQUIRED}  '
      f'well_reward={WELL_REWARD_SCR}  pretrial_min_steps={PRETRIAL_MIN_STEPS}')
print(f'RUN_ROOT={RUN_ROOT}')

_ALL_ARMS = ['full', 'B1_memoryless', 'B5_no_reward', 'B3_oracle', 'B2_stack', 'B4_switched']
if set(ARMS) != set(_ALL_ARMS):
    _skipped = [a for a in _ALL_ARMS if a not in ARMS]
    print(f'note: §10 runs {len(ARMS)}/6 arms; NOT running {_skipped} '
          f'(plan §11 M3 wants all six — add them to ARMS above or use the §13 headline recipe).')

# ── Yoked warm-start (§7.5): bootstrap past the exploration wall with a real rat ──
# The cold agent almost never reaches a goal (§9), so w/ψ never bootstrap. Seed the
# buffer with a real rat's goal-reaching trajectories (canonicalized to the agent's
# frame), imitate the policy, and anchor it while free-RL refines. (plan §3 knob 12)
YOKE_WARM_START      = True
YOKE_SUBJECT         = 'CM000'                 # in-roster PI+VC subject; canonical goal auto-derived
YOKE_ACTIONS_VARIANT = 'synthetic_pretrial'    # 'synthetic_pretrial' | 'real_pretrial'
YOKE_SF_UPDATES      = 60                       # off-policy SF/w updates on the seeded rat data
YOKE_BC_EPOCHS       = 8                        # behavioral-cloning epochs (Q-imitation of rat actions)
YOKE_ANCHOR_EVERY    = 3                        # re-apply 1 BC epoch every N free-train updates (retention)
YOKE_COMPARE_COLD    = True                     # also train a cold baseline for the comparison
YOKE_TEST_SESSIONS   = 2                         # held-out acquisition sessions for the no-yoking test
YOKE_FREE_EPISODES   = {'smoke': 20, 'demo': 80, 'headline': 200}[SCALE]
YOKE_FREE_MAX_STEPS  = 400
YOKE_DATASET_DIR     = REPO_ROOT / 'data' / 'yoked' / 'dataset'   # abs path (cwd-independent)

## 2. Successor-feature basis `φ`, pose encoder `e`, phase labels

Two *different* pose objects (plan §3, knobs 1–2):
- **`e(pose)`** — the repo-standard **60-D grid-cell code**, the compact spatial *input* to
  the heads. (`GridCellEncoder`, layout-invariant lookup.)
- **`φ(pose)`** — the **196-D one-hot pose** (49 floor cells × 4 headings) that is the
  successor-feature *target basis*. This is the exact basis the closed-form SR control solved
  the task with, so the SF-TD target stays well-conditioned.

The privileged **phase label** (train-only) comes from `env.session_phase` mapped through
`state_vectors._STATE_TYPE_TO_PHASE` to `{trial=0, pretrial=1, iti=2}`.

In [ ]:
def pose_to_phi_idx(x: int, y: int, d: int) -> int:
    """(x,y,dir) → index into the 196-D one-hot pose basis, or -1 if off-floor."""
    li = LOCATION_TO_INDEX.get((int(x), int(y)))
    return -1 if li is None else int(d) * NUM_LOCATIONS + li

def phi_onehot(idx: int) -> np.ndarray:
    v = np.zeros(PHI_DIM, dtype=np.float32)
    if 0 <= idx < PHI_DIM:
        v[idx] = 1.0
    return v

def phi_onehot_action(a: int) -> np.ndarray:
    v = np.zeros(NUM_ACTIONS, dtype=np.float32)
    if 0 <= a < NUM_ACTIONS:
        v[a] = 1.0
    return v

def env_phase(env) -> int:
    """Privileged, TRAIN-ONLY phase read: env.session_phase → {0,1,2}."""
    st = env.session_phase
    return PHASE_ITI if st is None else _STATE_TYPE_TO_PHASE.get(int(st), PHASE_ITI)

# well pose → well index {0:NE,1:NW,2:SW,3:SE}  (for the goal-identity probe)
WELL_XY_TO_IDX = {tuple(w): i for i, w in enumerate(WELL_LOCATIONS)}

ENCODER = GridCellEncoder()

# ── self-test: dims + coverage over a random walk ──
_env = CornerMazeEnv(session_type=SESSION_TYPE_ACQ, obs_mode='embedding',
                     agent_cue_goal_orientation=ORIENTATION, start_goal_location=GOAL_LOCATION,
                     max_steps=300)
_obs, _ = _env.reset(seed=0)
assert np.asarray(_obs['embedding']).shape == (OBS_DIM,)
assert ENCODER.encode(*_env.agent_pose).shape == (POSE_ENC_DIM,)
_miss = 0
for _ in range(300):
    m = _env.get_action_mask(); valid = [i for i in range(4) if m[i]]
    _o, _r, _t, _tr, _ = _env.step(int(np.random.choice(valid)))
    if pose_to_phi_idx(*_env.agent_pose) < 0: _miss += 1
    if _t or _tr: break
print(f'φ dim={PHI_DIM}  e dim={POSE_ENC_DIM}  obs dim={OBS_DIM}')
print(f'off-floor poses in 300-step walk: {_miss} (expect 0)')
print('phase map:', {k: v for k, v in _STATE_TYPE_TO_PHASE.items()})

## 3. Networks (plan §4)

```
 o_t ──┐
a_{t-1}┼─► GRU ─► h_t ─┐
 r_t ──┘               ├─► z_t=[e(pose_t), h_t] ─┬─► ψ(z_t,·) ∈ R^{A×dφ} ─┐
       pose_t ─► e(·) ─┘                          ├─► w(z_t) ∈ R^{dφ} ─────┼─► Q(z_t,a)=ψ·w
                                                   ├─► r̂_t = φ(z_t)·w(z_t)  │
                                                   ├─► c_head(h_t) ─► phase   (train-only aux)
                                                   └─► T_head(z_t,a) ─► p(pose_{t+1})  (aux)
```

`z_t = [e(pose_t), h_t]` fuses a spatial index with the inferred context. A **single**
action-conditioned SF head `ψ` (no per-context maps) is re-weighted by a **context-dependent**
`w(z_t)`: the same pose gets `w>0` when `h_t` has inferred "reward-active goal" and `w≈0` in
pretrial or for an error well. Orthogonal init throughout (repo convention).

In [ ]:
def _ortho(module: nn.Module, gain: float) -> nn.Module:
    for name, p in module.named_parameters():
        if 'weight' in name and p.dim() >= 2:
            nn.init.orthogonal_(p, gain=gain)
        elif 'bias' in name:
            nn.init.zeros_(p)
    return module

def _mlp(in_dim, hidden, out_dim, out_gain=1.0) -> nn.Sequential:
    return nn.Sequential(_ortho(nn.Linear(in_dim, hidden), math.sqrt(2)),
                         nn.ReLU(),
                         _ortho(nn.Linear(hidden, out_dim), out_gain))

def _phase_onehot_t(phase: int, device) -> torch.Tensor:
    v = torch.zeros(1, NUM_PHASES, device=device)
    if 0 <= phase < NUM_PHASES:
        v[0, phase] = 1.0
    return v


class RecurrentSFNet(nn.Module):
    """GRU context + SF head + context-dependent reward-weight head + aux heads."""
    def __init__(self, cfg: RecurrentSFConfig):
        super().__init__()
        self.cfg = cfg
        H, z = cfg.h_dim, cfg.z_dim
        n_maps = cfg.n_maps if cfg.switched_maps else 1

        if cfg.memory == 'gru' and H > 0:
            self.gru = nn.GRUCell(cfg.gru_in_dim, H); _ortho(self.gru, 1.0)
        elif cfg.memory == 'stack_k' and H > 0:
            self.stack_proj = _mlp(cfg.gru_in_dim * cfg.stack_k, cfg.hidden, H, out_gain=math.sqrt(2))

        self.sf_head = _mlp(z, cfg.hidden, n_maps * NUM_ACTIONS * PHI_DIM, out_gain=1.0)
        self.w_head  = _mlp(z, cfg.hidden, n_maps * PHI_DIM, out_gain=1.0)
        if cfg.switched_maps:
            self.gate_head = _mlp(z, cfg.hidden, n_maps, out_gain=0.01)
        if not cfg.tie_reward_head:
            self.r_head = _mlp(z, cfg.hidden, 1, out_gain=1.0)
        ctx_dim = H if H > 0 else z
        self.phase_head = _mlp(ctx_dim, cfg.hidden, NUM_PHASES, out_gain=1.0)   # train-only
        self.trans_head = _mlp(z + NUM_ACTIONS, cfg.hidden, PHI_DIM, out_gain=1.0)
        self.n_maps = n_maps

    def step_h(self, gru_input, h_prev):
        if self.cfg.memory == 'gru' and self.cfg.h_dim > 0:
            return self.gru(gru_input, h_prev)
        return None

    def context_from_stack(self, stacked):
        # tanh-bound the stack context to the GRU's (-1,1) range so z (hence ψ/w)
        # can't blow up under an unbounded linear projection.
        return torch.tanh(self.stack_proj(stacked))

    def make_z(self, e_pose, h, phase_onehot):
        parts = [e_pose]
        if h is not None:
            parts.append(h)
        if self.cfg.ctx_input == 'oracle':
            parts.append(phase_onehot)
        return torch.cat(parts, dim=-1)

    def heads(self, z):
        """psi (B,A,PHI), w (B,PHI), q (B,A) from agent state z."""
        B = z.shape[0]
        sf = self.sf_head(z).view(B, self.n_maps, NUM_ACTIONS, PHI_DIM)
        w  = self.w_head(z).view(B, self.n_maps, PHI_DIM)
        if self.cfg.switched_maps:
            gate = torch.softmax(self.gate_head(z), dim=-1)
            psi   = (sf * gate[:, :, None, None]).sum(1)
            w_eff = (w  * gate[:, :, None]).sum(1)
        else:
            psi, w_eff = sf[:, 0], w[:, 0]
        q = torch.einsum('bap,bp->ba', psi, w_eff)
        return {'psi': psi, 'w': w_eff, 'q': q}

    def reward_pred(self, z, phi_idx, w):
        """r̂_t. Tied: φ·w = w[pose_idx]. Untied: MLP(z)."""
        if self.cfg.tie_reward_head:
            valid = phi_idx.clamp(min=0)
            rhat = w.gather(1, valid[:, None]).squeeze(1)
            return torch.where(phi_idx >= 0, rhat, torch.zeros_like(rhat))
        return self.r_head(z).squeeze(-1)


# ── self-test: §4.4 shapes ──
_net = RecurrentSFNet(CFG).to(DEVICE)
_z = torch.zeros(3, CFG.z_dim, device=DEVICE)
_o = _net.heads(_z)
assert _o['psi'].shape == (3, NUM_ACTIONS, PHI_DIM)
assert _o['w'].shape == (3, PHI_DIM)
assert _o['q'].shape == (3, NUM_ACTIONS)
_ph = _net.phase_head(torch.zeros(3, CFG.h_dim or CFG.z_dim, device=DEVICE))
assert _ph.shape == (3, NUM_PHASES)
_nparams = sum(p.numel() for p in _net.parameters())
print(f'shapes OK — psi{tuple(_o["psi"].shape)}  w{tuple(_o["w"].shape)}  q{tuple(_o["q"].shape)}')
print(f'params = {_nparams:,}')

## 4. Sequence replay buffer + TBPTT windows (plan §6.2)

Whole episodes are stored; `sample_windows(B, L)` returns **boundary-respecting,
time-ordered** windows (never shuffle within a window, never cross an episode boundary).
Each window carries the stored recurrent state `h[s-1]` (detached) so temporal context
survives truncation.

In [ ]:
class Episode:
    """Per-step arrays for one finished episode. One record per env.step call:
    (o_t, e_t, φ_idx_t, a_{t-1}, r_{t-1}, phase_t, mask_t, a_t, r_t, done_t, term_t, h_t).
    `done` = episode ended (terminated OR truncated); `term` = TRUE termination
    (goal reached), used to zero the SF bootstrap. Truncation (time limit) is NOT
    terminal — its final transition is simply left unscored (no in-slice next)."""
    __slots__ = ('o','e','phi_idx','a_prev','r_prev','phase','mask','a','r','done','term','h','T')
    def __init__(self):
        for s in ('o','e','phi_idx','a_prev','r_prev','phase','mask','a','r','done','term','h'):
            setattr(self, s, [])
    def finalize(self):
        self.o=np.asarray(self.o,np.float32); self.e=np.asarray(self.e,np.float32)
        self.phi_idx=np.asarray(self.phi_idx,np.int64); self.a_prev=np.asarray(self.a_prev,np.int64)
        self.r_prev=np.asarray(self.r_prev,np.float32); self.phase=np.asarray(self.phase,np.int64)
        self.mask=np.asarray(self.mask,bool); self.a=np.asarray(self.a,np.int64)
        self.r=np.asarray(self.r,np.float32); self.done=np.asarray(self.done,bool)
        self.term=np.asarray(self.term,bool)
        self.h=(np.asarray(self.h,np.float32) if len(self.h) and self.h[0] is not None else None)
        self.T=len(self.a); return self


class SequenceReplayBuffer:
    def __init__(self, capacity): self.capacity=capacity; self.episodes=[]
    def push(self, ep):
        self.episodes.append(ep)
        if len(self.episodes) > self.capacity: self.episodes.pop(0)
    def __len__(self): return len(self.episodes)
    @property
    def total_steps(self): return sum(e.T for e in self.episodes)
    def sample_windows(self, batch, L, rng):
        out=[]
        if not self.episodes: return out
        for _ in range(batch):
            ep=self.episodes[rng.integers(len(self.episodes))]; T=ep.T
            s = 0 if T<=L else int(rng.integers(0, T-L))
            e = min(s+L+1, T)              # +1 for the in-slice bootstrap step
            out.append({'ep':ep,'s':s,'e':e})
        return out


class RunningStd:
    def __init__(self): self.n=0; self.mean=0.0; self.m2=0.0
    def update(self, xs):
        for x in xs.ravel():
            self.n+=1; d=x-self.mean; self.mean+=d/self.n; self.m2+=d*(x-self.mean)
    @property
    def std(self):
        return 1.0 if self.n<2 else max(math.sqrt(self.m2/(self.n-1)), 1e-4)


# ── self-test: windows respect boundaries ──
_b = SequenceReplayBuffer(10); _rng = np.random.default_rng(0)
for _T in (5, 40, 12):
    _ep = Episode()
    for _t in range(_T):
        for _s in ('o','e'): getattr(_ep,_s).append(np.zeros(60,np.float32))
        _ep.phi_idx.append(0); _ep.a_prev.append(0); _ep.r_prev.append(0.0); _ep.phase.append(0)
        _ep.mask.append([True]*5); _ep.a.append(0); _ep.r.append(0.0)
        _ep.done.append(_t==_T-1); _ep.term.append(_t==_T-1); _ep.h.append(np.zeros(CFG.h_dim,np.float32))
    _b.push(_ep.finalize())
for w in _b.sample_windows(20, 16, _rng):
    assert 0 <= w['s'] < w['e'] <= w['ep'].T and (w['e']-w['s']) <= 17
print(f'buffer self-test OK — {len(_b)} eps, {_b.total_steps} steps, windows in-bounds')

## 5. `RecurrentSFAgent` (plan §5 losses, §6 training)

Implements the `TrainableAgent` protocol so it plugs into `train.runner`. Four losses:
1. **SF-TD** `ψ(z_t,a_t) ← φ(z_t) + γ ψ⁻(z_{t+1}, a*)` (Q-learning target, masked; target nets, Polyak τ).
2. **Reward regression** `(r_t − φ(z_t)·w(z_t))²` — trains the *context-dependent* `w`.
3. **Next-pose prediction** (predictive-state grounding).
4. **Phase aux** `CE(c_head(h_t), phase_t)` — privileged, **train-only**, cosine-annealed.

> **Reward-scaling note (knob 10).** The plan's literal `REWARD_NORM=return_std` is *degenerate*
> here: the reward is bimodal (tiny step costs ≈1e-3 + a rare well reward ≈1.06), so the running
> per-step std ≈2e-4 is dominated by the step costs and would inflate the well reward ~5000×.
> We default to `reward_norm='fixed'` (divide by `WELL_REWARD_SCR`, so the well target ≈1.0) —
> the plan's *intent* (fix scale mismatch) without the pathology. `'return_std'` is available to
> reproduce the failure.

> **`w_from_phases` note (plan default `{TRIAL}`).** We read the phase label *before* each env step,
> so the goal reward (returned by that step) is attributed to the **TRIAL**-phase record that earned
> it — `{TRIAL}` therefore captures every goal reward (there is no "reward gets dropped" trap). We
> keep the plan's `{TRIAL}`-only discipline by default. Widening `w_from_phases` to all three phases
> is a sound alternative that *additionally* teaches `w≈0` in pretrial/iti (making the reward-machine
> gating explicit); exposure — the one real contamination source for a static `w` — is never trained
> here, so neither choice risks contamination.

In [ ]:
class RecurrentSFAgent(TrainableAgent):
    def __init__(self, cfg: RecurrentSFConfig, encoder: GridCellEncoder):
        self.cfg = cfg; self.encoder = encoder
        self.device = torch.device(cfg.device)
        torch.manual_seed(cfg.seed)
        self.net = RecurrentSFNet(cfg).to(self.device)
        self.target = copy.deepcopy(self.net).to(self.device)
        for p in self.target.parameters(): p.requires_grad_(False)
        self.opt = torch.optim.Adam(self.net.parameters(), lr=cfg.lr)
        self.buffer = SequenceReplayBuffer(cfg.buffer_capacity)
        self.rstd = RunningStd()
        self.rng = np.random.default_rng(cfg.seed)
        self._n_updates = 0; self._frozen = False; self._episodes_since_update = 0
        self._reset_episode_state()

    # ---- per-episode recurrent state (plan §6.3) ----
    def begin_episode(self): self._reset_episode_state()
    def _reset_episode_state(self):
        self._h = (torch.zeros(1, self.cfg.h_dim, device=self.device) if self.cfg.use_memory else None)
        self._a_prev = 0; self._r_prev = 0.0; self._cur = Episode(); self._stack = []

    # ---- schedules ----
    @property
    def _temp(self):
        f = min(1.0, self._n_updates / max(1, self.cfg.temp_anneal_updates))
        return self.cfg.temp_start + f * (self.cfg.temp_end - self.cfg.temp_start)
    @property
    def _lambda_ctx(self):
        f = min(1.0, self._n_updates / max(1, self.cfg.ctx_anneal_updates))
        hi, lo = self.cfg.lambda_ctx, self.cfg.lambda_ctx_floor
        return lo + 0.5 * (hi - lo) * (1 + math.cos(math.pi * f))
    def _rdiv(self):
        m = self.cfg.reward_norm
        return float(WELL_REWARD_SCR) if m=='fixed' else (self.rstd.std if m=='return_std' else 1.0)
    def _norm_r(self, r): return r / self._rdiv()
    def set_frozen(self, frozen): self._frozen = frozen

    def _gru_input(self, o, a_prev, r_prev):
        parts = [o.astype(np.float32), phi_onehot_action(a_prev)]
        if self.cfg.gru_input != 'no_reward':
            parts.append(np.array([self._norm_r(r_prev)], np.float32))
        return torch.from_numpy(np.concatenate(parts)).float().unsqueeze(0).to(self.device)

    # ---- TrainableAgent API ----
    def select_action(self, state, action_mask):
        o = np.asarray(state['embedding'], np.float32)
        x, y, d = state['pose']; phase = int(state['phase'])
        e = self.encoder.encode(x, y, d).astype(np.float32)
        phi_idx = pose_to_phi_idx(x, y, d)
        gin = self._gru_input(o, self._a_prev, self._r_prev)
        with torch.no_grad():
            if self.cfg.memory == 'gru' and self.cfg.use_memory:
                h = self.net.step_h(gin, self._h)
            elif self.cfg.memory == 'stack_k' and self.cfg.use_memory:
                self._push_stack(gin); h = self.net.context_from_stack(self._stacked_tensor())
            else:
                h = None
            e_t = torch.from_numpy(e).unsqueeze(0).to(self.device)
            z = self.net.make_z(e_t, h, _phase_onehot_t(phase, self.device))
            q = self.net.heads(z)['q'].squeeze(0).cpu().numpy().astype(np.float64)
        if self.cfg.memory == 'gru':
            self._h = h

        masked = q.copy()
        valid = [i for i, ok in enumerate(action_mask) if ok] or [0]
        for i in range(NUM_ACTIONS):
            if i not in valid: masked[i] = -np.inf

        if self._frozen:
            action = int(np.argmax(masked))
        elif self.rng.random() < self.cfg.eps_floor:
            action = int(valid[self.rng.integers(len(valid))])
        else:
            zl = masked / max(self._temp, 1e-6)
            zl = zl - np.nanmax(zl[np.isfinite(zl)])
            p = np.where(np.isfinite(zl), np.exp(zl), 0.0)
            action = (int(valid[self.rng.integers(len(valid))]) if p.sum() <= 0
                      else int(self.rng.choice(NUM_ACTIONS, p=p / p.sum())))

        self._pending = dict(o=o, e=e, phi_idx=phi_idx, phase=phase,
                             mask=list(action_mask[:NUM_ACTIONS]),
                             a_prev=self._a_prev, r_prev=self._r_prev,
                             h=(self._h.squeeze(0).cpu().numpy() if self._h is not None else None))
        return action, {'q': q}

    def add_experience(self, state, action, reward, done, **kw):
        p = self._pending; c = self._cur
        c.o.append(p['o']); c.e.append(p['e']); c.phi_idx.append(p['phi_idx'])
        c.a_prev.append(p['a_prev']); c.r_prev.append(p['r_prev']); c.phase.append(p['phase'])
        c.mask.append(p['mask']); c.a.append(int(action)); c.r.append(float(reward))
        c.done.append(bool(done)); c.term.append(bool(kw.get('terminated', done))); c.h.append(p['h'])
        self._a_prev = int(action); self._r_prev = float(reward)
        if done:
            self.rstd.update(np.asarray(c.r, np.float32))
            self.buffer.push(c.finalize()); self._episodes_since_update += 1
            self._reset_episode_state()

    def advance_frozen(self, action, reward, done):
        self._a_prev = int(action); self._r_prev = float(reward)
        if done: self._reset_episode_state()

    def is_ready_to_update(self):
        return (not self._frozen and len(self.buffer) >= self.cfg.min_episodes_to_update
                and self._episodes_since_update >= 1)

    def update(self, next_state=None, next_done=None):
        self._episodes_since_update = 0
        logs = [self._tbptt_step() for _ in range(self.cfg.updates_per_collect)]
        return {k: float(np.mean([lg[k] for lg in logs])) for k in logs[0]}

    # ---- TBPTT core (plan §6.2) ----
    def _tbptt_step(self):
        cfg = self.cfg
        windows = self.buffer.sample_windows(cfg.batch_windows, cfg.tbptt_l, self.rng)
        tot = {'psi':0.,'r':0.,'trans':0.,'ctx':0.}; n = {'psi':0,'r':0,'trans':0,'ctx':0}
        loss = torch.zeros((), device=self.device)
        for win in windows:
            ep, s, e = win['ep'], win['s'], win['e']; L = e - s
            o = torch.from_numpy(ep.o[s:e]).to(self.device)
            e_pose = torch.from_numpy(ep.e[s:e]).to(self.device)
            a_prev = torch.from_numpy(ep.a_prev[s:e]).to(self.device)
            r_prev = torch.from_numpy(ep.r_prev[s:e]).to(self.device)
            rdiv = self._rdiv()
            gin_parts = [o, F.one_hot(a_prev, NUM_ACTIONS).float()]
            if cfg.gru_input != 'no_reward':
                gin_parts.append((r_prev / rdiv)[:, None])
            gin = torch.cat(gin_parts, dim=-1)

            if cfg.memory == 'gru' and cfg.use_memory:
                h = (torch.zeros(cfg.h_dim, device=self.device) if (s == 0 or ep.h is None)
                     else torch.from_numpy(ep.h[s-1]).to(self.device))
                hs = []; ht = h
                for t in range(L):
                    ht = self.net.gru(gin[t:t+1], ht.unsqueeze(0)).squeeze(0); hs.append(ht)
                h_seq = torch.stack(hs)
            elif cfg.memory == 'stack_k' and cfg.use_memory:
                h_seq = self._stack_forward(ep, s, e, gin)
            else:
                h_seq = None

            phase = torch.from_numpy(ep.phase[s:e]).to(self.device)
            z = self.net.make_z(e_pose, h_seq, F.one_hot(phase, NUM_PHASES).float())
            out = self.net.heads(z); psi, w = out['psi'], out['w']

            phi_idx = torch.from_numpy(ep.phi_idx[s:e]).to(self.device)
            a = torch.from_numpy(ep.a[s:e]).to(self.device)
            r = torch.from_numpy(ep.r[s:e]).to(self.device)
            done = torch.from_numpy(ep.done[s:e].astype(np.float32)).to(self.device)
            term = torch.from_numpy(ep.term[s:e].astype(np.float32)).to(self.device)
            mask = torch.from_numpy(ep.mask[s:e]).to(self.device)

            phi = torch.zeros(L, PHI_DIM, device=self.device)
            vphi = phi_idx >= 0
            phi[torch.arange(L, device=self.device)[vphi], phi_idx[vphi].clamp(min=0)] = 1.0

            with torch.no_grad():
                ot = self.target.heads(z.detach())
                q_t = ot['q'].masked_fill(~mask, -1e9); a_star = q_t.argmax(-1)
                psi_t = ot['psi']

            # (1) SF-TD
            psi_loss = torch.zeros((), device=self.device); npsi = 0
            for t in range(L):
                if bool(term[t]):
                    target = phi[t]               # TRUE terminal (goal reached): no future
                elif t + 1 < L:
                    target = phi[t] + cfg.gamma * psi_t[t+1, int(a_star[t+1])]
                else:
                    continue        # window end / time-limit truncation: no in-slice next → unscored
                psi_loss = psi_loss + F.mse_loss(psi[t, a[t]], target); npsi += 1
            if npsi: psi_loss = psi_loss / npsi; tot['psi'] += psi_loss.item(); n['psi'] += 1

            # (2) reward regression (context-dependent w)
            rhat = self.net.reward_pred(z, phi_idx, w)
            r_target = r / rdiv
            wsel = torch.zeros(L, dtype=torch.bool, device=self.device)
            for ph in cfg.w_from_phases: wsel |= (phase == ph)
            wsel &= vphi
            r_loss = F.mse_loss(rhat[wsel], r_target[wsel]) if wsel.any() else torch.zeros((), device=self.device)
            if wsel.any(): tot['r'] += r_loss.item(); n['r'] += 1

            # (3) next-pose prediction
            trans_loss = torch.zeros((), device=self.device)
            if cfg.lambda_trans > 0 and L > 1:
                logits = self.net.trans_head(torch.cat([z, F.one_hot(a, NUM_ACTIONS).float()], -1))
                nxt = phi_idx[1:]; vnxt = (nxt >= 0) & (~done[:-1].bool())
                if vnxt.any():
                    trans_loss = F.cross_entropy(logits[:-1][vnxt], nxt[vnxt])
                    tot['trans'] += trans_loss.item(); n['trans'] += 1

            # (4) phase aux (privileged, train-only)
            ctx_loss = torch.zeros((), device=self.device)
            if cfg.lambda_ctx > 0:
                ph_logits = self.net.phase_head(h_seq if h_seq is not None else z)
                ctx_loss = F.cross_entropy(ph_logits, phase); tot['ctx'] += ctx_loss.item(); n['ctx'] += 1

            loss = loss + (cfg.lambda_psi*psi_loss + cfg.lambda_r*r_loss
                           + cfg.lambda_trans*trans_loss + self._lambda_ctx*ctx_loss)

        loss = loss / max(1, len(windows))
        self.opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(self.net.parameters(), cfg.grad_clip)
        self.opt.step(); self._polyak(); self._n_updates += 1
        return {'loss': float(loss.item()), **{k: tot[k]/max(1,n[k]) for k in tot},
                'temp': self._temp, 'lambda_ctx': self._lambda_ctx}

    def _polyak(self):
        tau = self.cfg.tau
        with torch.no_grad():
            for tp, p in zip(self.target.parameters(), self.net.parameters()):
                tp.mul_(1 - tau).add_(tau * p)

    # ---- stack_k (B2) helpers ----
    def _push_stack(self, gin):
        self._stack.append(gin.squeeze(0).cpu().numpy())
        if len(self._stack) > self.cfg.stack_k: self._stack.pop(0)
    def _stacked_tensor(self):
        k = self.cfg.stack_k
        pad = [np.zeros(self.cfg.gru_in_dim, np.float32)] * (k - len(self._stack))
        arr = np.concatenate(pad + self._stack)
        return torch.from_numpy(arr).float().unsqueeze(0).to(self.device)
    def _gin_row(self, ep, j):
        parts = [torch.from_numpy(ep.o[j]).to(self.device),
                 F.one_hot(torch.tensor(int(ep.a_prev[j]), device=self.device), NUM_ACTIONS).float()]
        if self.cfg.gru_input != 'no_reward':
            parts.append(torch.tensor([ep.r_prev[j] / self._rdiv()], device=self.device).float())
        return torch.cat(parts)
    def _stack_forward(self, ep, s, e, gin):
        L = e - s; k = self.cfg.stack_k; rows = []
        for t in range(L):
            feats = []
            for j in range(s+t-k+1, s+t+1):
                if j < 0: feats.append(torch.zeros(self.cfg.gru_in_dim, device=self.device))
                elif j >= s: feats.append(gin[j-s])
                else: feats.append(self._gin_row(ep, j))
            rows.append(torch.cat(feats))
        return self.net.context_from_stack(torch.stack(rows))

    # ---- io ----
    def save(self, path):
        if not path.endswith('.pt'): path += '.pt'
        torch.save({'net': self.net.state_dict(), 'cfg': self.cfg.__dict__}, path)
    def load(self, path):
        if not path.endswith('.pt'): path += '.pt'
        d = torch.load(path, map_location=self.device)
        self.net.load_state_dict(d['net']); self.target.load_state_dict(d['net'])


# ── M0 smoke: one TBPTT step + episode-boundary reset test (plan §6.3) ──
def _make_env(session_type=SESSION_TYPE_ACQ, max_steps=MAX_STEPS):
    return CornerMazeEnv(session_type=session_type, obs_mode='embedding',
                         agent_cue_goal_orientation=ORIENTATION,
                         start_goal_location=GOAL_LOCATION, max_steps=max_steps)

def run_episode(env, agent, *, train, enable_pause=False):
    obs, _ = env.reset(); agent.set_frozen(not train); agent.begin_episode()
    done = False; ep_reward = 0.0; steps = 0
    while not done:
        mask = env.get_action_mask(); mask[4] = enable_pause
        state = {'embedding': np.asarray(obs['embedding'], np.float32),
                 'pose': env.agent_pose, 'phase': env_phase(env)}
        action, info = agent.select_action(state, mask)
        obs, reward, term, trunc, _ = env.step(action)
        done = bool(term or trunc); ep_reward += float(reward); steps += 1
        if train: agent.add_experience(state, action, reward, done, terminated=bool(term), **info)
        else:     agent.advance_frozen(action, reward, done)
    return {'reward': ep_reward, 'steps': steps}

_a = RecurrentSFAgent(RecurrentSFConfig(tbptt_l=16, batch_windows=4,
                                        min_episodes_to_update=2, updates_per_collect=1), ENCODER)
_env2 = _make_env(max_steps=150)
for _ in range(3): run_episode(_env2, _a, train=True)
_log = _a.update()
print('one TBPTT step OK →', {k: round(v,4) for k,v in _log.items()})

# boundary-reset test: h after reset is exactly zeros, independent of prior episode content
_a.begin_episode(); _h0 = _a._h.clone()
with torch.no_grad():
    _a._h = _a.net.step_h(_a._gru_input(np.random.randn(60).astype(np.float32), 3, 1.5), _a._h)
assert torch.any(_a._h != _h0)          # h changed after a step
_a.begin_episode()
assert torch.all(_a._h == 0)            # reset → zeros, no leakage across the boundary
print('episode-boundary GRU reset test OK  (M0 ✓)')

## 6. Env rollout + runner hooks (plan §6.1, §8)

`build_harness(cfg)` returns the four callables `run_session_sequence` needs. The agent is
created **once per seed** and reused across acquisition + probe phases (like the PPO notebooks).
`train_fn` runs the collect→update cadence explicitly and caps acquisition at `MAX_ACQ_EPISODES`
so a non-acquiring run terminates (the runner then emits `killed_at.json` and skips probes).

In [ ]:
def build_harness(make_cfg, *, max_acq_episodes=MAX_ACQ_EPISODES, loss_sink=None):
    """make_cfg(seed)->RecurrentSFConfig. Returns (state, make_env, train_fn, frozen_fn, save_fn, reset_seed)."""
    state = {'agent': None, 'seed': None}

    def make_env(session_type):
        env = _make_env(session_type)
        return env, env                        # no wrapper needed for embedding obs

    def train_fn(env):
        agent = state['agent']
        is_acq = 'acquisition' in (getattr(env, 'session_type', '') or '').lower()
        cap = max_acq_episodes if is_acq else 3
        for _ in range(cap):
            run_episode(env, agent, train=True)
            if agent.is_ready_to_update():
                lg = agent.update()
                if loss_sink is not None: loss_sink.append({'seed': state['seed'], **lg})
            if getattr(env, 'training_criterion_met', False):
                break

    def frozen_fn(env):
        run_episode(env, state['agent'], train=False)

    def save_fn(path):
        state['agent'].save(path)

    def reset_seed(seed):
        state['agent'] = RecurrentSFAgent(make_cfg(seed), ENCODER); state['seed'] = seed

    return state, make_env, train_fn, frozen_fn, save_fn, reset_seed

print('harness builder ready')

## 7. Train the full model (via `run_session_sequence`)

Runs the canonical harness on the acquisition context (plus any `PROBE_SESSION_TYPES`),
one pass per seed, writing the repo-standard artifacts (`episode_data.parquet`,
`curves.parquet`, `trial_metrics.parquet`, model checkpoints, `run_config.json`). The trained
agents are kept in memory for the probe / metric / diagnostic sections below.

In [ ]:
RUN_ROOT.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S')
FULL_RUN_DIR = RUN_ROOT / f'full_{timestamp}'

full_agents = []          # trained agent per seed (for §8/§9/§12)
full_losses = []          # per-update loss dicts across seeds
full_curves = []          # per-episode learning-curve rows
seed_results = []

session_types = [SESSION_TYPE_ACQ] + list(PROBE_SESSION_TYPES)
state, make_env, train_fn, frozen_fn, save_fn, reset_seed = build_harness(
    lambda seed: RecurrentSFConfig(**{**CFG.__dict__, 'seed': seed}),
    loss_sink=full_losses)

for seed in range(N_SEEDS):
    print(f'\n=== full model — seed {seed} ===')
    seed_dir = FULL_RUN_DIR / f'seed_{seed}'; seed_dir.mkdir(parents=True, exist_ok=True)
    reset_seed(seed)
    t0 = time.time()
    result = run_session_sequence(
        session_types=session_types,
        make_env=make_env, train_fn=train_fn, frozen_fn=frozen_fn, save_fn=save_fn,
        save_data_path=str(seed_dir / 'episode_data.parquet'),
        model_save_dir=str(seed_dir), seed=seed, run_dir=str(seed_dir),
        eval_episodes_per_probe=3,
        run_config_extra={'run_name': RUN_NAME, 'model': 'recurrent_sf',
                          'experiment_notes': EXPERIMENT_NOTES, 'scale': SCALE,
                          'session_types': session_types, 'cfg': CFG.__dict__})
    full_agents.append(state['agent'])
    # per-episode learning curve from the acquisition episode rows
    if result.df is not None and len(result.df):
        acq = result.df[(result.df['session_type'] == SESSION_TYPE_ACQ)
                        & (result.df['policy_mode'] == 'updating')]
        for i, (_, row) in enumerate(acq.iterrows()):
            ts = row['trial_scores']
            score = (sum(ts)/len(ts)) if isinstance(ts, list) and ts else 0.0
            full_curves.append({'seed': seed, 'episode': i, 'mean_trial_score': score,
                                'perfect_trials': int(sum(ts) if isinstance(ts, list) else 0),
                                'steps': int(row.get('total_steps', 0)),
                                'reward': float(row.get('total_reward', 0.0))})
    print(f'  decision={result.decision.value}  scores={[round(s,3) for s in result.scores]}  '
          f'buffer={len(state["agent"].buffer)} eps  {time.time()-t0:.1f}s')
    seed_results.append({'seed': seed, 'decision': result.decision.value, 'scores': result.scores})

import pandas as pd
curves_df = pd.DataFrame(full_curves)
print(f'\ndone: {len(full_agents)} seed(s) → {FULL_RUN_DIR}')
if len(curves_df):
    print('acquisition mean-trial-score (last 5 eps, seed 0):',
          [round(x,3) for x in curves_df[curves_df.seed==0]['mean_trial_score'].tail(5)])

## 7.5 Yoked warm-start from a subject (plan §3 knob 12) — bootstrapping past the exploration wall

§9 shows the cold agent almost never stumbles onto a goal, so `w`/`ψ` never get a reward to
bootstrap from. Here we **seed the replay buffer with a real rat's goal-reaching trajectories**.

The recipe below was validated empirically (goals reached in 30 eval rollouts, subject CM000):
`cold=2 · SF/w-only warm=4 · +behavioral-cloning=7 · BC then naïve free-RL=0 (forgotten) · **BC + anchored free-RL=20**`.
So three ingredients matter, in order:
1. **Seed + off-policy SF/`w`.** Replay the rat's actions through the env to collect reward-bearing
   episodes; SF-TD is off-policy (Q-learning target), so it can learn `w` (goal value) from them.
   Modest on its own — `ψ` learned from the *rat's* policy isn't the agent's.
2. **Behavioral cloning (Q-imitation).** Make the rat's action the argmax of masked `Q` — this
   transfers the *navigation policy*, not just the value.
3. **Anchoring.** Naïve free-RL *forgets* the cloned policy; re-applying one BC epoch every few
   updates retains it while RL refines. This is what makes it stick.

**Train/test split (no leakage).** We hold out the subject's **last `YOKE_TEST_SESSIONS` (=2)
acquisition sessions** — the rat's graduation sessions. Warm-start (rat replay + BC) and free-RL
use only the **earlier** sessions; the reported score is the **frozen agent navigating the 2
held-out sessions with no yoking** (its own actions, unseen `trial_configs`). The cold baseline is
free-trained on the same earlier sessions and evaluated on the same held-out sessions. So the
headline is *generalization to unseen sessions*, not replay of trained ones.

**Frame:** each rat session is `canonicalize_session`'d (cue → north) and replayed with the rat's
own `trial_configs`, so warm-start, free-training, and the held-out test all share an identical
observation/cue/goal frame by construction (the subject's canonical goal, auto-derived — CM000's
is NW). Requires the yoked dataset (`data/yoked/dataset/`); the cell no-ops with a message if absent.

The three ingredients that matter, in order (validated on the held-out sessions): **(1)** seed +
off-policy SF/`w` (values the goal, but `ψ` is the rat's not the agent's); **(2)** behavioral
cloning — Q-imitation of the rat's actions — transfers the *navigation policy*; **(3)** anchoring —
re-apply BC every few free-RL updates, else naïve free-RL forgets the cloned policy.

In [ ]:
import itertools
import matplotlib.pyplot as plt

yoked_agent = None
GOAL_IDX_TO_NAME = {0: 'NE', 1: 'SE', 2: 'SW', 3: 'NW'}

def _yoke_available():
    try:
        from corner_maze_rl.data.load import YokedPaths
        YokedPaths.from_dir(dataset_dir=YOKE_DATASET_DIR, actions_variant=YOKE_ACTIONS_VARIANT).assert_exist()
        return True
    except Exception as e:
        print(f'yoked dataset unavailable ({type(e).__name__}) — skipping §7.5. '
              f'Build it with corner-maze-build-dataset, or set YOKE_WARM_START=False.')
        return False

if YOKE_WARM_START and _yoke_available():
    from corner_maze_rl.data.load import (YokedPaths, load_subjects, load_sessions,
                                          load_actions_for_session, get_sessions_for_subject)
    from corner_maze_rl.data.canonical import canonicalize_session

    def load_canonical_sessions(subject, variant=YOKE_ACTIONS_VARIANT):
        paths = YokedPaths.from_dir(dataset_dir=YOKE_DATASET_DIR, actions_variant=variant)
        subj = load_subjects(paths); sess = load_sessions(paths)
        row = subj[subj['subject_name'] == subject]
        if len(row) == 0:
            raise ValueError(f'subject {subject!r} not in dataset')
        row = row.iloc[0]
        ss = get_sessions_for_subject(sess, int(row['subject_id']))
        acq = ss[ss['session_phase'] == 'Acquisition']
        out = []
        for _, sr in acq.iterrows():
            tc = json.loads(sr['trial_configs'])
            adf = load_actions_for_session(paths, int(sr['session_id']))
            rtc, rdf, R = canonicalize_session(tc, adf)
            out.append((rtc, rdf.reset_index(drop=True)))
        return out, row

    def make_yoked_env(rtc, max_steps):
        return CornerMazeEnv(session_type='PI+VC f2 acquisition', obs_mode='embedding',
                             agent_cue_goal_orientation='N/NE',
                             start_goal_location=GOAL_IDX_TO_NAME[int(rtc[0][2])],
                             trial_configs=rtc, max_steps=max_steps)

    def replay_into_buffer(agent, rtc, rdf):
        """Force the rat's actions through the env; store the trajectory (with the
        agent's GRU context) in the buffer. Returns #goal rewards seen."""
        env = make_yoked_env(rtc, max_steps=len(rdf) + 50)
        obs, _ = env.reset(); agent.set_frozen(False); agent.begin_episode()
        acts = rdf['action'].to_numpy(); got = 0
        for t in range(len(acts)):
            mask = env.get_action_mask(); mask[4] = False
            state = {'embedding': np.asarray(obs['embedding'], np.float32),
                     'pose': env.agent_pose, 'phase': env_phase(env)}
            _, info = agent.select_action(state, mask)      # advance h + set _pending (action ignored)
            a = int(acts[t])
            obs, r, term, trunc, _ = env.step(a)            # FORCE the rat's action
            done = bool(term or trunc)
            agent.add_experience(state, a, r, done, terminated=bool(term), **info)
            if r > 0.5: got += 1
            if done: break
        return got

    def bc_pretrain(agent, episodes, n_epochs, grad_clip=1.0):
        """Behavioral cloning through the Q-head: rat action = argmax of masked Q."""
        dev = agent.device
        for _ in range(n_epochs):
            for ep in episodes:
                h = torch.zeros(1, agent.cfg.h_dim, device=dev); zs = []
                for t in range(ep.T):
                    o = torch.from_numpy(ep.o[t]).float().to(dev)
                    a_oh = F.one_hot(torch.tensor(int(ep.a_prev[t]), device=dev), NUM_ACTIONS).float()
                    parts = [o, a_oh]
                    if agent.cfg.gru_input != 'no_reward':
                        parts.append(torch.tensor([ep.r_prev[t] / agent._rdiv()], device=dev).float())
                    h = agent.net.gru(torch.cat(parts)[None], h)
                    e = torch.from_numpy(ep.e[t]).float().to(dev)[None]
                    zs.append(agent.net.make_z(e, h, _phase_onehot_t(int(ep.phase[t]), dev)))
                Q = agent.net.heads(torch.cat(zs))['q'].masked_fill(
                    ~torch.from_numpy(ep.mask).to(dev), -1e9)
                loss = F.cross_entropy(Q, torch.from_numpy(ep.a).to(dev))
                agent.opt.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(agent.net.parameters(), grad_clip); agent.opt.step()

    def free_train_yoked(agent, rtc_list, n_episodes, yoked_eps=None, bc_every=0,
                         max_steps=YOKE_FREE_MAX_STEPS):
        cyc = itertools.cycle(rtc_list); k = 0
        for _ in range(n_episodes):
            run_episode(make_yoked_env(next(cyc), max_steps), agent, train=True)
            if agent.is_ready_to_update():
                agent.update(); k += 1
                if yoked_eps and bc_every and (k % bc_every == 0):
                    bc_pretrain(agent, yoked_eps, n_epochs=1)

    def eval_heldout(agent, test_rtcs, frozen=True, reps=1, max_steps=YOKE_FREE_MAX_STEPS):
        """Run the (frozen) agent on HELD-OUT sessions with NO yoking — its own actions on
        unseen trial_configs. Counts goals reached vs trials available. Deterministic when
        frozen ⇒ reps=1 suffices; use reps>1 with frozen=False for a stochastic estimate."""
        goals = trig = pre = possible = 0
        for rtc in test_rtcs:
            for _ in range(reps):
                possible += len(rtc)                 # one goal available per trial
                env = make_yoked_env(rtc, max_steps)
                obs, _ = env.reset(); agent.set_frozen(frozen); agent.begin_episode()
                done = False; prev = None
                while not done:
                    mask = env.get_action_mask(); mask[4] = False
                    state = {'embedding': np.asarray(obs['embedding'], np.float32),
                             'pose': env.agent_pose, 'phase': env_phase(env)}
                    a, _ = agent.select_action(state, mask)
                    obs, r, term, trunc, _ = env.step(a); done = bool(term or trunc)
                    ph = env_phase(env)
                    if ph == PHASE_PRETRIAL and prev != PHASE_PRETRIAL: pre += 1
                    if prev == PHASE_PRETRIAL and ph == PHASE_TRIAL: trig += 1
                    if r > 0.5: goals += 1
                    agent.advance_frozen(a, r, done); prev = ph
        return dict(goals=goals, possible=possible, solve_rate=round(goals / max(1, possible), 3),
                    trigger_rate=round(trig / max(1, pre), 3))

    _base = {k: getattr(CFG, k) for k in
             ('tbptt_l', 'batch_windows', 'min_episodes_to_update', 'updates_per_collect')}
    sessions, subj_row = load_canonical_sessions(YOKE_SUBJECT)
    n_test = min(YOKE_TEST_SESSIONS, max(1, len(sessions) - 1))
    train_sessions, test_sessions = sessions[:-n_test], sessions[-n_test:]
    rtc_train = [s[0] for s in train_sessions]           # each = one session's [arm,cue,goal,tag] trials
    rtc_test  = [s[0] for s in test_sessions]
    goal_name = GOAL_IDX_TO_NAME[int(rtc_train[0][0][2])]  # session 0 → trial 0 → goal field
    rat_rew_train = [int(rdf['rewarded'].sum()) for _, rdf in train_sessions]
    print(f'subject {YOKE_SUBJECT} (group={subj_row["training_group"]}): canonical goal={goal_name}')
    print(f'  TRAIN: {len(train_sessions)} sessions yoked (rat rewards {rat_rew_train})')
    print(f'  TEST : {len(test_sessions)} HELD-OUT sessions, {[len(r) for r in rtc_test]} trials each '
          f'(frozen agent, no yoking)')

    results = {}
    if YOKE_COMPARE_COLD:                                 # cold: free-RL on TRAIN only, no yoking
        cold = RecurrentSFAgent(RecurrentSFConfig(seed=0, **_base), ENCODER)
        free_train_yoked(cold, rtc_train, YOKE_FREE_EPISODES)
        results['cold'] = eval_heldout(cold, rtc_test)
        print(f'  [cold]           held-out {results["cold"]}')

    # warm: seed TRAIN replays → SF/w + BC → anchored free-RL on TRAIN
    warm = RecurrentSFAgent(RecurrentSFConfig(seed=0, **_base), ENCODER)
    yoked_eps, seeded = [], 0
    for rtc, rdf in train_sessions:
        seeded += replay_into_buffer(warm, rtc, rdf); yoked_eps.append(warm.buffer.episodes[-1])
    print(f'  seeded buffer from TRAIN: {len(warm.buffer)} episodes, {warm.buffer.total_steps} steps, '
          f'{seeded} goal rewards')
    for _ in range(YOKE_SF_UPDATES):
        if warm.is_ready_to_update(): warm.update()
    bc_pretrain(warm, yoked_eps, YOKE_BC_EPOCHS)
    results['warm_prefree'] = eval_heldout(warm, rtc_test)
    print(f'  [warm pre-free]  held-out {results["warm_prefree"]}')
    free_train_yoked(warm, rtc_train, YOKE_FREE_EPISODES, yoked_eps=yoked_eps, bc_every=YOKE_ANCHOR_EVERY)
    results['warm'] = eval_heldout(warm, rtc_test)                       # frozen = deployed policy
    print(f'  [warm post-free] held-out (frozen)      {results["warm"]}')
    warm_expl = eval_heldout(warm, rtc_test, frozen=False, reps=3)       # exploratory = policy potential
    print(f'  [warm post-free] held-out (exploratory) {warm_expl}')
    yoked_agent = warm

    # grouped bars: held-out trials-solved AND trigger-cross rate by condition.
    # (trigger-crossing = the "reach the start position / activate the reward regime" behavior
    #  the whole model is about; it transfers even when goal-navigation only weakly does.)
    order = [k for k in ('cold', 'warm_prefree', 'warm') if k in results]
    labels = {'cold': 'cold\n(no yoking)', 'warm_prefree': 'warm BC\n(pre free-RL)', 'warm': 'warm+anchor\n(post free-RL)'}
    x = np.arange(len(order)); w = 0.38
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    ax.bar(x - w/2, [results[k]['solve_rate'] for k in order], w,
           label='trials solved (goals / available)', color='#2ca02c')
    ax.bar(x + w/2, [results[k]['trigger_rate'] for k in order], w,
           label='trigger-cross rate', color='#4a90d9')
    for i, k in enumerate(order):
        ax.text(x[i] - w/2, results[k]['solve_rate'], f"{results[k]['goals']}/{results[k]['possible']}",
                ha='center', va='bottom', fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels([labels[k] for k in order])
    ax.set_ylabel('rate on HELD-OUT sessions'); ax.set_ylim(0, 1.08); ax.legend(fontsize=8)
    ax.set_title(f'Yoked warm-start ({YOKE_SUBJECT}→{goal_name}) — {len(test_sessions)} HELD-OUT sessions, '
                 f'frozen agent, no yoking')
    plt.tight_layout(); plt.savefig(RUN_ROOT / f'yoke_{YOKE_SUBJECT}_{SCALE}.png', dpi=110); plt.show()
else:
    print('§7.5 skipped (YOKE_WARM_START=False or dataset unavailable).')

## 8. Representational probes on `h_t` (plan §10.2) — **the headline**

The decisive test: does the recurrent state `h_t` actually encode the hidden context? We fit
**convex linear probes** (logistic regression, L2) on *frozen* `h_t` captured by re-running the
network over the replay experience (which visits all phases — the deployed argmax policy can get
stuck in pretrial, so buffer experience is the principled probe-data source):

- **Phase probe** `h_t → {pretrial, trial, iti}` — low error ⇒ `h_t` linearly encodes the
  reward-machine state.
- **Reward-active probe** `h_t → {trial-active, not}`.
- **Steps-since-trigger** — linear read-out (regression) of the trigger-crossing counter.
- **Goal-identity probe** `h_t → which well was the goal` (only where trials completed).

Error is compared against the chance (majority-class) floor.

> **Is this circular?** No. The phase-aux loss (#4) is applied to **every arm equally**, including
> the memoryless B1. B1 still fails the phase probe because `e(pose)` *cannot* represent a phase
> that pose alone doesn't reveal — the supervision can't install information the input lacks. So
> the **full ≪ B1** gap isolates the contribution of *memory*, not of the supervision. For the
> strongest form of the claim, set `lambda_ctx=0` (recipes §13) and confirm `h_t` still encodes
> phase from the SF + predictive-state losses alone. Also note **B5 (no-reward) ≈ full** here is
> *expected*: the pretrial→trial flip is physically visible (barriers open), so phase is not a
> *reward-only* distinction. The reward-history necessity shows up on goal-identity under matched
> visuals (M2 cue/goal contexts), not on phase.

In [ ]:
class LinearProbe:
    """Convex linear probe: L2-regularized logistic regression fit with LBFGS."""
    def __init__(self, in_dim, n_classes, l2=1e-3, max_iter=300, device='cpu'):
        self.device = torch.device(device)
        self.lin = nn.Linear(in_dim, n_classes).to(self.device)
        self.l2, self.max_iter = l2, max_iter
    def fit(self, X, y):
        X = torch.as_tensor(X, dtype=torch.float32, device=self.device)
        y = torch.as_tensor(y, dtype=torch.long, device=self.device)
        opt = torch.optim.LBFGS(self.lin.parameters(), lr=0.5, max_iter=self.max_iter)
        def closure():
            opt.zero_grad()
            loss = F.cross_entropy(self.lin(X), y) + self.l2 * self.lin.weight.pow(2).sum()
            loss.backward(); return loss
        opt.step(closure); return self
    def error(self, X, y):
        X = torch.as_tensor(X, dtype=torch.float32, device=self.device)
        with torch.no_grad():
            return float(np.mean(self.lin(X).argmax(-1).cpu().numpy() != np.asarray(y)))

class LinearRegProbe:
    """Convex linear regression (closed-form ridge) — for steps-since-trigger."""
    def __init__(self, l2=1e-2): self.l2 = l2; self.W = None
    def fit(self, X, y):
        X = np.asarray(X, np.float64); y = np.asarray(y, np.float64)
        Xb = np.hstack([X, np.ones((len(X),1))])
        A = Xb.T@Xb + self.l2*np.eye(Xb.shape[1]); self.W = np.linalg.solve(A, Xb.T@y); return self
    def r2(self, X, y):
        X = np.asarray(X, np.float64); y = np.asarray(y, np.float64)
        pred = np.hstack([X, np.ones((len(X),1))])@self.W
        ss_res = ((y-pred)**2).sum(); ss_tot = ((y-y.mean())**2).sum()+1e-9
        return float(1 - ss_res/ss_tot)


def collect_context_labels(agent, max_episodes=60):
    """Re-run the FROZEN net over replay episodes → (H, phase, reward_active, sst, goal).

    H          : context vector per step (h_t for GRU/stack; e(pose) for memoryless)
    phase      : {0,1,2}
    reward_active: 1 if phase==TRIAL else 0
    sst        : steps-since-trigger within the current trial (-1 outside trial)
    goal       : well idx {0..3} for a completed trial's steps, else -1
    """
    net = agent.net; H, PH, SST, GOAL, EPID = [], [], [], [], []
    for ep_i, ep in enumerate(agent.buffer.episodes[-max_episodes:]):
        T = ep.T
        # per-step context
        ctx = np.zeros((T, agent.cfg.h_dim or POSE_ENC_DIM), np.float32)
        with torch.no_grad():
            if agent.cfg.use_memory and agent.cfg.memory == 'gru':
                h = torch.zeros(agent.cfg.h_dim, device=agent.device)
                for t in range(T):
                    parts = [torch.from_numpy(ep.o[t]).to(agent.device),
                             F.one_hot(torch.tensor(int(ep.a_prev[t])), NUM_ACTIONS).float().to(agent.device)]
                    if agent.cfg.gru_input != 'no_reward':
                        parts.append(torch.tensor([ep.r_prev[t]/agent._rdiv()], device=agent.device).float())
                    h = net.gru(torch.cat(parts).unsqueeze(0), h.unsqueeze(0)).squeeze(0)
                    ctx[t] = h.cpu().numpy()
            elif agent.cfg.use_memory and agent.cfg.memory == 'stack_k':
                stack = []
                for t in range(T):
                    parts = [torch.from_numpy(ep.o[t]).to(agent.device),
                             F.one_hot(torch.tensor(int(ep.a_prev[t])), NUM_ACTIONS).float().to(agent.device)]
                    if agent.cfg.gru_input != 'no_reward':
                        parts.append(torch.tensor([ep.r_prev[t]/agent._rdiv()], device=agent.device).float())
                    stack.append(torch.cat(parts).cpu().numpy())
                    if len(stack) > agent.cfg.stack_k: stack.pop(0)
                    pad = [np.zeros(agent.cfg.gru_in_dim, np.float32)]*(agent.cfg.stack_k-len(stack))
                    ct = net.context_from_stack(torch.from_numpy(np.concatenate(pad+stack)).float().unsqueeze(0).to(agent.device))
                    ctx[t] = ct.squeeze(0).cpu().numpy()
            else:
                ctx = ep.e.copy()
        # steps-since-trigger + goal labels from phase/reward sequence
        sst = np.full(T, -1, np.int64); goal = np.full(T, -1, np.int64)
        counter = -1; seg_start = None
        for t in range(T):
            ph = int(ep.phase[t])
            if ph == PHASE_TRIAL:
                counter = 0 if counter < 0 else counter + 1
                if seg_start is None: seg_start = t
                sst[t] = counter
            else:
                counter = -1
            # close a trial segment on leaving TRIAL (or episode end)
            is_last = (t == T-1)
            if (ph != PHASE_TRIAL or is_last) and seg_start is not None:
                seg_end = t if ph != PHASE_TRIAL else t+1
                # did a reward fire in [seg_start, seg_end)?
                rew_idx = [k for k in range(seg_start, seg_end) if ep.r[k] > 0.5]
                if rew_idx:
                    k = rew_idx[0]
                    # The reward record stores the PRE-step pose (the corner/approach cell),
                    # not the well — the agent occupies the goal well on the NEXT record.
                    # So read the goal well index from k+1 (fall back to k).
                    gidx = _phi_idx_to_well(int(ep.phi_idx[k+1])) if k+1 < T else -1
                    if gidx < 0: gidx = _phi_idx_to_well(int(ep.phi_idx[k]))
                    if gidx >= 0: goal[seg_start:seg_end] = gidx   # label the trial's steps with its goal
                seg_start = None
        H.append(ctx); PH.append(ep.phase.copy()); SST.append(sst); GOAL.append(goal)
        EPID.append(np.full(T, ep_i, np.int64))
    H = np.concatenate(H); PH = np.concatenate(PH); SST = np.concatenate(SST); GOAL = np.concatenate(GOAL)
    RA = (PH == PHASE_TRIAL).astype(np.int64)
    return dict(H=H, phase=PH, reward_active=RA, sst=SST, goal=GOAL, ep_id=np.concatenate(EPID))


# invert pose_to_phi_idx for well cells: recover (x,y) then map to a well index
_IDX_TO_XY = {li: xy for xy, li in LOCATION_TO_INDEX.items()}
def _phi_idx_to_well(phi_idx):
    if phi_idx < 0: return -1
    li = phi_idx % NUM_LOCATIONS
    return WELL_XY_TO_IDX.get(_IDX_TO_XY.get(li), -1)


def probe_report(agent, tag, max_episodes=60, seed=0):
    data = collect_context_labels(agent, max_episodes)
    H = data['H']; n = len(H)
    # EPISODE-level split: adjacent steps within an episode are highly
    # autocorrelated, so a per-step random split would leak near-duplicates
    # across train/test and understate the probe error. Split whole episodes.
    eids = data['ep_id']; uniq = np.unique(eids)
    rng = np.random.default_rng(seed); rng.shuffle(uniq)
    n_tr = max(1, int(0.7 * len(uniq)))
    tr_eps = set(uniq[:n_tr].tolist())
    tr = np.array([e in tr_eps for e in eids]); te = ~tr
    rows = {}
    def _clf(H_, y_, k, tr_, te_):
        if tr_.sum() < 5 or te_.sum() < 5: return None
        chance = 1 - max((y_[te_] == c).mean() for c in np.unique(y_[te_]))
        err = LinearProbe(H_.shape[1], k, device=DEVICE).fit(H_[tr_], y_[tr_]).error(H_[te_], y_[te_])
        return dict(error=err, chance=chance)
    # phase {trial,pretrial,iti}
    r = _clf(H, data['phase'], NUM_PHASES, tr, te)
    if r: rows['phase'] = dict(**r, n=n)
    # reward-active {trial, not}
    if len(np.unique(data['reward_active'])) == 2:
        r = _clf(H, data['reward_active'], 2, tr, te)
        if r: rows['reward_active'] = dict(**r, n=n)
    # steps-since-trigger (linear regression on trial steps)
    m = data['sst'] >= 0
    if (m & tr).sum() > 20 and (m & te).sum() > 10:
        r2 = LinearRegProbe().fit(H[m & tr], data['sst'][m & tr].astype(np.float32)) \
                             .r2(H[m & te], data['sst'][m & te].astype(np.float32))
        rows['steps_since_trigger'] = dict(r2=r2, n=int(m.sum()))
    # goal identity (only where trials completed)
    g = data['goal']; gm = g >= 0
    if (gm & tr).sum() > 20 and (gm & te).sum() > 10 and len(np.unique(g[gm])) >= 2:
        r = _clf(H, g, 4, gm & tr, gm & te)
        if r: rows['goal_identity'] = dict(**r, n=int(gm.sum()))
    print(f'\n[{tag}]  (n={n} steps, {len(uniq)} episodes)')
    for k, v in rows.items():
        print(f'   {k:20s} ' + '  '.join(f'{kk}={vv:.3f}' if isinstance(vv, float) else f'{kk}={vv}'
                                          for kk, vv in v.items()))
    return rows

full_probe = probe_report(full_agents[0], 'full model — h_t')

## 9. Behavioral context metrics (plan §10.1)

Computed from the trained agent's replay experience (phase transitions + reward events):
- **trigger-crossing rate** — fraction of pretrial segments where the agent left the dead-end,
  held the trigger, and flipped to `TRIAL` (earning no reward en route);
- **trial success rate** — of activated trials, how many reached the goal well;
- **activation-to-reward latency** — mean trial-phase steps before the goal.

These describe the *deployed* behavior; at modest budgets they can be near-zero — reported honestly.

In [ ]:
def behavioral_metrics(agent, max_episodes=80):
    eps = agent.buffer.episodes[-max_episodes:]
    n_pretrial_seg = n_cross = n_trials = n_success = 0
    lat = []
    for ep in eps:
        prev = None; in_pretrial = False; trial_len = 0; in_trial = False
        for t in range(ep.T):
            ph = int(ep.phase[t])
            if ph == PHASE_PRETRIAL and not in_pretrial:
                in_pretrial = True; n_pretrial_seg += 1
            if prev == PHASE_PRETRIAL and ph == PHASE_TRIAL:
                n_cross += 1
            if ph == PHASE_TRIAL:
                if not in_trial: in_trial = True; trial_len = 0; n_trials += 1
                trial_len += 1
                if ep.r[t] > 0.5:
                    n_success += 1; lat.append(trial_len); in_trial = False
            else:
                if prev == PHASE_TRIAL: in_trial = False
                if ph != PHASE_PRETRIAL: in_pretrial = False
            prev = ph
    return dict(
        n_episodes=len(eps),
        trigger_cross_rate=(n_cross / n_pretrial_seg) if n_pretrial_seg else 0.0,
        n_trigger_crossings=n_cross,
        trial_success_rate=(n_success / n_trials) if n_trials else 0.0,
        n_goal_reached=n_success,
        activation_latency=float(np.mean(lat)) if lat else float('nan'))

full_behav = behavioral_metrics(full_agents[0])
print('full-model behavioral metrics (seed 0):')
for k, v in full_behav.items():
    print(f'   {k:22s} {v}')

## 10. Baseline matrix (plan §9) — the arm comparison

Each arm is a minimal config delta of the full model, trained on the same acquisition context
for `BASELINE_EPISODES` episodes × `N_SEEDS` seeds (a light direct loop — faster than the full
harness and independent of whether the agent acquires). We report the **phase-probe error** with
IQM + a stratified-bootstrap 95% CI over seeds. The plan predicts **full ≲ B3 oracle**, both
**≪ B1 memoryless**, with **B5 (no reward)** matching the full model where contexts differ
visually but degrading where they differ only in reward outcome.

In [ ]:
def bootstrap_ci(vals, n_boot=2000, seed=0):
    vals = np.asarray(vals, float)
    if len(vals) == 0: return (float('nan'),)*3
    if len(vals) == 1: return float(vals[0]), float(vals[0]), float(vals[0])
    rng = np.random.default_rng(seed)
    boots = [np.mean(rng.choice(vals, len(vals), replace=True)) for _ in range(n_boot)]
    return float(np.mean(vals)), float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))

def train_arm(arm, seed, n_episodes=BASELINE_EPISODES, max_steps=MAX_STEPS):
    agent = RecurrentSFAgent(arm_config(arm, seed, **{
        k: getattr(CFG, k) for k in ('tbptt_l','batch_windows','min_episodes_to_update','updates_per_collect')}),
        ENCODER)
    env = _make_env(max_steps=max_steps)
    for _ in range(n_episodes):
        run_episode(env, agent, train=True)
        if agent.is_ready_to_update(): agent.update()
    return agent

arm_records = []
print(f'training {len(ARMS)} arms × {N_SEEDS} seeds × {BASELINE_EPISODES} episodes ...')
for arm in ARMS:
    t0 = time.time()
    for seed in range(N_SEEDS):
        ag = train_arm(arm, seed)
        pr = probe_report(ag, f'{arm} s{seed}', max_episodes=50, seed=seed)
        bm = behavioral_metrics(ag, max_episodes=50)
        arm_records.append({'arm': arm, 'seed': seed,
                            'phase_err': pr['phase']['error'], 'phase_chance': pr['phase']['chance'],
                            'reward_active_err': pr.get('reward_active', {}).get('error', float('nan')),
                            'sst_r2': pr.get('steps_since_trigger', {}).get('r2', float('nan')),
                            'trigger_cross_rate': bm['trigger_cross_rate'],
                            'trial_success_rate': bm['trial_success_rate']})
    print(f'  {arm}: {time.time()-t0:.1f}s')

arm_df = pd.DataFrame(arm_records)
summary = []
for arm in ARMS:
    sub = arm_df[arm_df.arm == arm]
    m, lo, hi = bootstrap_ci(sub['phase_err'].values)
    summary.append({'arm': arm, 'phase_err_iqm': m, 'ci_lo': lo, 'ci_hi': hi,
                    'phase_chance': sub['phase_chance'].mean(),
                    'reward_active_err': sub['reward_active_err'].mean(),
                    'trigger_cross_rate': sub['trigger_cross_rate'].mean(),
                    'trial_success_rate': sub['trial_success_rate'].mean(),
                    'n_seeds': len(sub)})
summary_df = pd.DataFrame(summary).sort_values('phase_err_iqm').reset_index(drop=True)
print('\n=== PHASE-PROBE ERROR by arm (lower = h_t encodes context better) ===')
print(summary_df.to_string(index=False, float_format=lambda x: f'{x:.3f}'))
summary_df.to_parquet(RUN_ROOT / f'arm_summary_{SCALE}.parquet')

## 11. Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# (a) learning curve (acquisition mean-trial-score per episode)
ax = axes[0, 0]
if len(curves_df):
    for seed in sorted(curves_df.seed.unique()):
        s = curves_df[curves_df.seed == seed]
        ax.plot(s['episode'], s['mean_trial_score'], alpha=.5, label=f'seed {seed}')
    ax.axhline(CRITERION_SCORE_THRESHOLD, ls='--', c='k', lw=1, label=f'criterion {CRITERION_SCORE_THRESHOLD}')
    ax.legend(fontsize=8)
ax.set_title('Full model — acquisition learning curve'); ax.set_xlabel('episode')
ax.set_ylabel('mean trial score'); ax.set_ylim(-0.02, 1.02)

# (b) loss components over updates (full model)
ax = axes[0, 1]
if full_losses:
    ldf = pd.DataFrame(full_losses)
    for comp, c in [('psi','C0'), ('r','C1'), ('trans','C2'), ('ctx','C3')]:
        if comp in ldf:
            ax.plot(np.arange(len(ldf)), ldf[comp].values, c=c, alpha=.7, label=comp)
    ax.legend(fontsize=8); ax.set_yscale('log')
ax.set_title('Loss components (full model)'); ax.set_xlabel('update'); ax.set_ylabel('loss (log)')

# (c) phase-probe error by arm (headline)
ax = axes[1, 0]
if len(summary_df):
    y = np.arange(len(summary_df))
    err_lo = summary_df['phase_err_iqm'] - summary_df['ci_lo']
    err_hi = summary_df['ci_hi'] - summary_df['phase_err_iqm']
    ax.barh(y, summary_df['phase_err_iqm'], xerr=[err_lo, err_hi], color='C0', alpha=.8, capsize=3)
    ax.plot(summary_df['phase_chance'], y, 'kx', label='chance')
    ax.set_yticks(y); ax.set_yticklabels(summary_df['arm']); ax.invert_yaxis()
    ax.legend(fontsize=8)
ax.set_title('Phase-probe error by arm  (lower ⇒ h_t encodes context)'); ax.set_xlabel('probe error')

# (d) behavioral: trigger-cross & trial-success by arm
ax = axes[1, 1]
if len(summary_df):
    x = np.arange(len(summary_df)); w = 0.38
    ax.bar(x - w/2, summary_df['trigger_cross_rate'], w, label='trigger-cross rate', color='C2')
    ax.bar(x + w/2, summary_df['trial_success_rate'], w, label='trial-success rate', color='C4')
    ax.set_xticks(x); ax.set_xticklabels(summary_df['arm'], rotation=20, ha='right', fontsize=8)
    ax.legend(fontsize=8)
ax.set_title('Behavioral metrics by arm'); ax.set_ylabel('rate')

plt.tight_layout(); plt.savefig(RUN_ROOT / f'figures_{SCALE}.png', dpi=110); plt.show()
print(f'saved → {RUN_ROOT / f"figures_{SCALE}.png"}')

## 12. Diagnostics — context-dependent `w(z_t)` (plan §4.3, §10.1 metric 3)

The whole trick is that a **single** `ψ` is re-weighted by a context-dependent `w`. Here we hold
the pose fixed and vary only the inferred context `h_t` (pretrial vs trial), and read out
`w(z_t)[pose]` = the predicted reward at that pose. If the agent has learned the reward machine,
the **same goal-well pose** should get `w≈0` under a pretrial context and `w>0` under a trial
context — no map was switched, only the reward weights.

In [ ]:
def mean_w_at_pose(agent, xy_dir, h_vectors):
    """Mean predicted reward w(z)[pose] at a FIXED pose, injecting REAL context
    vectors h_t sampled from the replay buffer. Holds e(pose) at the goal and
    varies only the inferred context — the counterfactual "if the agent were here
    but believed it was in phase X, what reward would it predict?"."""
    x, y, d = xy_dir
    phi_idx = pose_to_phi_idx(x, y, d)
    if phi_idx < 0 or len(h_vectors) == 0:
        return float('nan')
    e = torch.from_numpy(agent.encoder.encode(x, y, d).astype(np.float32)).unsqueeze(0).to(agent.device)
    vals = []
    with torch.no_grad():
        for h in h_vectors:
            ht = torch.from_numpy(np.asarray(h, np.float32)).unsqueeze(0).to(agent.device) if agent.cfg.use_memory else None
            z = agent.net.make_z(e, ht, _phase_onehot_t(PHASE_TRIAL, agent.device))
            vals.append(agent.net.heads(z)['w'].squeeze(0)[phi_idx].item())
    return float(np.mean(vals))

ag = full_agents[0]
goal_pose = (WELL_LOCATIONS[0][0], WELL_LOCATIONS[0][1], 0)   # NE well
_d = collect_context_labels(ag, 60)
_Hpre = _d['H'][_d['phase'] == PHASE_PRETRIAL][:400]
_Htrl = _d['H'][_d['phase'] == PHASE_TRIAL][:400]
w_pre = mean_w_at_pose(ag, goal_pose, _Hpre)
w_trl = mean_w_at_pose(ag, goal_pose, _Htrl)
print('context-dependent reward readout  mean w(z)[pose]  at the NE goal well')
print('(real h_t sampled from the buffer; e(pose) held fixed at the goal):')
print(f'   pretrial-context h_t : {w_pre:+.4f}   (n={len(_Hpre)})')
print(f'   trial-context   h_t : {w_trl:+.4f}   (n={len(_Htrl)})')
print(f'   → trial − pretrial gap = {w_trl - w_pre:+.4f}')
if full_behav['n_goal_reached'] < 3:
    print('   ⚠ the agent reached the goal only %d time(s) at this budget, so w at the goal is '
          'barely trained — expect the gap to be small/noisy until goal-reward experience '
          'accumulates (scale up in §13). The METHOD is the point here; the effect grows with M1.'
          % full_behav['n_goal_reached'])
else:
    print('   (positive gap ⇒ the SAME pose reads as rewarding only when h_t infers reward-active — '
          'no map was switched, only the context-dependent w)')

# phase-confusion of the trained phase-aux head over the buffer
data = collect_context_labels(ag, 40)
Hc = torch.from_numpy(data['H']).float().to(ag.device)
with torch.no_grad():
    if ag.cfg.use_memory:
        pred = ag.net.phase_head(Hc).argmax(-1).cpu().numpy()
    else:
        pred = np.full(len(data['phase']), -1)
if (pred >= 0).all():
    conf = np.zeros((NUM_PHASES, NUM_PHASES), int)
    for tru, pr in zip(data['phase'], pred): conf[tru, pr] += 1
    print('\nphase-aux confusion (rows=true {trial,pretrial,iti}, cols=pred):')
    print(conf)

## 13. Recipes — preset scales & how to push toward M1–M5

```python
# --- SMOKE (≈1 min): prove every path runs ---
SCALE = 'smoke'

# --- DEMO (default, ≈5–10 min): clear phase-probe separation across arms ---
SCALE = 'demo'

# --- HEADLINE (≈1 hr+): more seeds/episodes for CI'd figures ---
SCALE = 'headline'; N_SEEDS = 5

# --- memory-stress variant (no cue): the GRU must carry the goal, not just the phase ---
SESSION_TYPE_ACQ = 'PI acquisition'            # no cue → goal is hidden; expect B5/B1 to fall off

# --- full six-arm matrix (plan §9) ---
ARMS = ['full','B1_memoryless','B5_no_reward','B3_oracle','B2_stack','B4_switched']

# --- reproduce the plan's degenerate reward-norm (educational) ---
CFG = RecurrentSFConfig(reward_norm='return_std')   # watch the reward loss explode

# --- push toward solving navigation (M1): longer episodes + budget ---
PRESETS['headline'].update(MAX_ACQ_EPISODES=1500, MAX_STEPS=676)

# --- yoked warm-start from a subject (§7.5): the fix for the exploration wall ---
YOKE_WARM_START = True; YOKE_SUBJECT = 'CM000'   # any in-roster PI+VC subject (goal auto-derived)
YOKE_ANCHOR_EVERY = 3        # lower ⇒ stronger retention of the cloned policy through free-RL
YOKE_WARM_START = False      # ...or turn it off to see the cold exploration wall (few goals)
```

**Yoked warm-start (validated, CM000):** goals reached in 30 eval rollouts — `cold=2`,
`SF/w-only=4`, `+behavioral-cloning=7`, `BC then naïve free-RL=0 (forgotten)`,
**`BC + anchored free-RL=20`**. Seeding a real rat's canonicalized trajectories, cloning the
policy, and anchoring it through free-RL is what clears the sparse-reward exploration wall.

**What "done" looks like** (plan §13): `h_t` phase/reward-active probes show the predicted
structure, full ≪ B5 on reward-only-distinguishable contexts, and the six-arm table reproduces
`full ≳ B3 > B2 ≈ B4 > B1`. Promoting this notebook's inline classes into
`src/corner_maze_rl/models/recurrent_sf.py` (+ `tests/test_recurrent_sf.py`) per the plan §7
manifest is the natural next step once the science checks out here.